In [ ]:
import transformers, torch
print("transformers:", transformers.__version__, "| torch:", torch.__version__)


transformers: 4.57.2 | torch: 2.9.0+cu126


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Dict, List

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)


from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/COLAB_ARCHIVOS/MET_Proyecto"

BASE_DATA   = os.path.join(PROJECT_ROOT, "Data")
SALIDAS     = os.path.join(BASE_DATA, "Salidas")
CSV_PATH    = os.path.join(BASE_DATA, "fakes1000.csv")

REPORT_DIR  = os.path.join(SALIDAS, "reportes_transformers")
MODELOS_DIR = os.path.join(SALIDAS, "modelos", "Transformers")
GRAF_DIR    = os.path.join(MODELOS_DIR, "graficas")

os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(MODELOS_DIR, exist_ok=True)
os.makedirs(GRAF_DIR, exist_ok=True)

print("Usando GPU:", torch.cuda.is_available())

!nvidia-smi -L || true

MODEL_NAME   = "dccuchile/bert-base-spanish-wwm-cased"
MAX_LENGTH   = 256
EPOCHS       = 3
SEED         = 42
LRS          = [5e-5, 3e-5, 2e-5]
BATCH_SIZES  = [8, 16]

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


@dataclass
class TextDataset(Dataset):
    encodings: Dict[str, torch.Tensor]
    labels: List[int]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def load_texts_and_labels(csv_path: str, text_col: str = "Text", label_col: str = "class"):
    df = pd.read_csv(csv_path)
    texts = df[text_col].astype(str).tolist()
    labels = df[label_col].astype(int).tolist()
    return texts, labels


def tokenize_texts(tokenizer, texts: List[str], max_length: int):
    return tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors="pt",
    )


def compute_clf_metrics(y_true, y_pred):
    return {
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
    }


def build_compute_metrics_fn():
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {
            "accuracy":  accuracy_score(labels, preds),
            "precision": precision_score(labels, preds, zero_division=0),
            "recall":    recall_score(labels, preds, zero_division=0),
            "f1":        f1_score(labels, preds, zero_division=0),
        }
    return compute_metrics


def predict_with_trainer(trainer: "Trainer", dataset: Dataset):
    preds_output = trainer.predict(dataset)
    logits = preds_output.predictions
    y_pred = logits.argmax(axis=-1)
    return y_pred


def make_training_args(output_dir, lr, bs, epochs, seed):
    import inspect
    from transformers import TrainingArguments
    import torch

    bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
    fp16 = torch.cuda.is_available() and not bf16

    desired = dict(
        output_dir=output_dir,
        learning_rate=lr,
        per_device_train_batch_size=bs,
        per_device_eval_batch_size=bs,
        num_train_epochs=epochs,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=50,
        report_to=[],
        seed=seed,
        fp16=fp16,
        bf16=bf16,
        warmup_ratio=0.06,
        optim="adamw_torch",
    )

    sig = inspect.signature(TrainingArguments.__init__)
    allowed = {k: v for k, v in desired.items() if k in sig.parameters}

    has_eval = ("evaluation_strategy" in sig.parameters)
    has_save = ("save_strategy" in sig.parameters)
    has_load_best = ("load_best_model_at_end" in sig.parameters)

    if has_eval and has_save:
        allowed["evaluation_strategy"] = "epoch"
        allowed["save_strategy"] = "epoch"
        if has_load_best:
            allowed["load_best_model_at_end"] = True
    else:
        if has_save:
            allowed["save_strategy"] = "no"
        if has_load_best:
            allowed["load_best_model_at_end"] = False
        allowed.pop("metric_for_best_model", None)
        allowed.pop("greater_is_better", None)

    return TrainingArguments(**allowed)



def train_and_select_best(tokenizer, X_train, y_train, X_val, y_val):
    import os
    from transformers import AutoModelForSequenceClassification, Trainer
    import torch, numpy as np

    enc_train = tokenize_texts(tokenizer, X_train, MAX_LENGTH)
    enc_val   = tokenize_texts(tokenizer, X_val, MAX_LENGTH)
    ds_train  = TextDataset(enc_train, y_train)
    ds_val    = TextDataset(enc_val, y_val)

    best_f1 = -1.0
    best_cfg = None
    best_model_dir = None

    def _pick_or_save_model_dir(trainer, output_dir):

        best_ckpt = getattr(trainer.state, "best_model_checkpoint", None)
        if best_ckpt and os.path.exists(os.path.join(best_ckpt, "config.json")):
            return best_ckpt

        try:
            ckpts = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
            if ckpts:
                ckpt = sorted(ckpts, key=lambda x: int(x.split("-")[-1]))[-1]
                ckpt_dir = os.path.join(output_dir, ckpt)
                if os.path.exists(os.path.join(ckpt_dir, "config.json")):
                    return ckpt_dir
        except Exception:
            pass

        final_dir = os.path.join(output_dir, "final")
        os.makedirs(final_dir, exist_ok=True)
        trainer.save_model(final_dir)
        tokenizer.save_pretrained(final_dir)
        return final_dir

    for lr in LRS:
        for bs in BATCH_SIZES:
            run_name   = f"beto_lr{lr}_bs{bs}"
            output_dir = os.path.join(MODELOS_DIR, run_name)
            os.makedirs(output_dir, exist_ok=True)

            model = AutoModelForSequenceClassification.from_pretrained(
                MODEL_NAME,
                num_labels=2
            ).to(device)
            args  = make_training_args(output_dir, lr, bs, EPOCHS, SEED)

            trainer = Trainer(
                model=model,
                args=args,
                train_dataset=ds_train,
                eval_dataset=ds_val,
                tokenizer=tokenizer,
                compute_metrics=build_compute_metrics_fn(),
            )

            trainer.train()
            eval_res = trainer.evaluate(eval_dataset=ds_val)
            f1_val   = float(eval_res.get("f1", 0.0))
            print(f"[{run_name}] F1 val = {f1_val:.4f}")


            log_history = trainer.state.log_history

            model_dir_to_use = _pick_or_save_model_dir(trainer, output_dir)

            if f1_val > best_f1:
                best_f1 = f1_val
                best_cfg = {"lr": lr, "batch_size": bs}
                best_model_dir = model_dir_to_use


                hist_path = os.path.join(REPORT_DIR, "history_beto.npy")
                np.save(hist_path, np.array(log_history, dtype=object))
                print("Historial del mejor BETO guardado en:", hist_path)

            del trainer, model
            torch.cuda.empty_cache()

    print("Mejor config:", best_cfg, " | Mejor F1 val:", best_f1)
    print("best_model_dir:", best_model_dir)

    assert os.path.exists(os.path.join(best_model_dir, "config.json")), \
        f"No hay config.json en {best_model_dir}"
    return best_cfg, best_model_dir




def plot_beto_curves(history_file="history_beto.npy"):
    history_path = os.path.join(REPORT_DIR, history_file)
    if not os.path.exists(history_path):
        print(f"No se encontró el historial de BETO en {history_path}")
        return

    logs = np.load(history_path, allow_pickle=True)

    train_epochs, train_loss = [], []
    eval_epochs, eval_loss, eval_f1 = [], [], []


    for rec in logs:
        if not isinstance(rec, dict):
            continue


        if "loss" in rec and "epoch" in rec and "eval_loss" not in rec:
            train_epochs.append(rec["epoch"])
            train_loss.append(rec["loss"])


        if "eval_loss" in rec and "epoch" in rec:
            eval_epochs.append(rec["epoch"])
            eval_loss.append(rec["eval_loss"])
            if "eval_f1" in rec:
                eval_f1.append(rec["eval_f1"])


    if train_epochs or eval_epochs:
        plt.figure()
        if train_epochs:
            plt.plot(train_epochs, train_loss, label="Train loss", marker="o")
        if eval_epochs:
            plt.plot(eval_epochs, eval_loss, label="Val loss", marker="s")
        plt.title("BETO - Loss por época")
        plt.xlabel("Época")
        plt.ylabel("Loss")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        out_path = os.path.join(GRAF_DIR, "loss_beto.png")
        plt.savefig(out_path, dpi=150)
        plt.close()
        print("Guardado:", out_path)


    if eval_epochs and eval_f1:
        plt.figure()
        plt.plot(eval_epochs, eval_f1, label="Val F1", marker="o")
        plt.title("BETO - F1 de validación por época")
        plt.xlabel("Época")
        plt.ylabel("F1-score")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        out_path = os.path.join(GRAF_DIR, "f1_beto.png")
        plt.savefig(out_path, dpi=150)
        plt.close()
        print("Guardado:", out_path)



def main():
    texts, labels = load_texts_and_labels(CSV_PATH, "Text", "class")


    X_temp, X_test, y_temp, y_test = train_test_split(
        texts, labels, test_size=0.20, random_state=SEED, stratify=labels
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.125, random_state=SEED, stratify=y_temp
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    best_cfg, best_model_dir = train_and_select_best(tokenizer, X_train, y_train, X_val, y_val)


    def _has_config(path: str) -> bool:
        return os.path.exists(os.path.join(path, "config.json"))

    def _try_load_model(dirpath: str):
        return AutoModelForSequenceClassification.from_pretrained(
            dirpath,
            num_labels=2
        ).to(device)

    def _resolve_model_dir(base_dir: str) -> str:
        if _has_config(base_dir):
            return base_dir

        alt = os.path.join(base_dir, "final")
        if _has_config(alt):
            print(f"[fallback] Usando subcarpeta: {alt}")
            return alt

        try:
            ckpts = [d for d in os.listdir(base_dir) if d.startswith("checkpoint-")]
            if ckpts:
                ckpt = sorted(ckpts, key=lambda x: int(x.split("-")[-1]))[-1]
                ckpt_dir = os.path.join(base_dir, ckpt)
                if _has_config(ckpt_dir):
                    print(f"[fallback] Usando último checkpoint: {ckpt_dir}")
                    return ckpt_dir
        except Exception:
            pass

        return base_dir

    resolved_dir = _resolve_model_dir(best_model_dir)
    try:
        best_model = _try_load_model(resolved_dir)
        best_model_dir = resolved_dir
    except Exception as e:
        raise RuntimeError(
            f"No se pudo cargar un modelo válido desde '{best_model_dir}'. "
            f"Probé: base, 'final' y último 'checkpoint-*'. Detalle: {e}"
        )

    # Dataset para train completo y test
    enc_train = tokenize_texts(tokenizer, X_train, MAX_LENGTH)
    enc_test  = tokenize_texts(tokenizer, X_test,  MAX_LENGTH)
    ds_train  = TextDataset(enc_train, y_train)
    ds_test   = TextDataset(enc_test,  y_test)

    pred_trainer = Trainer(model=best_model, tokenizer=tokenizer)
    y_pred_train = predict_with_trainer(pred_trainer, ds_train)
    y_pred_test  = predict_with_trainer(pred_trainer, ds_test)

    m_train = compute_clf_metrics(y_train, y_pred_train)
    m_test  = compute_clf_metrics(y_test,  y_pred_test)

    cm = confusion_matrix(y_test, y_pred_test)
    cm_name = "cm_beto.npy"
    np.save(os.path.join(REPORT_DIR, cm_name), cm)

    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
    else:
        tn = fp = fn = tp = None

    row = {
        "modelo": "beto_finetuned",
        "acc_train": m_train["accuracy"],
        "prec_train": m_train["precision"],
        "rec_train": m_train["recall"],
        "f1_train": m_train["f1"],
        "acc_test": m_test["accuracy"],
        "prec_test": m_test["precision"],
        "rec_test": m_test["recall"],
        "f1_test": m_test["f1"],
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
        "cm_path": cm_name,
        "best_lr": best_cfg["lr"],
        "best_batch_size": best_cfg["batch_size"],
        "model_dir": os.path.relpath(best_model_dir, start=SALIDAS),
    }

    report_path = os.path.join(REPORT_DIR, "report_transformers.csv")
    pd.DataFrame([row]).to_csv(report_path, index=False, encoding="utf-8")

    print("\n Reporte guardado en:", report_path)
    print(pd.DataFrame([row]))
    print(" Mejor modelo en:", best_model_dir)

    df = pd.read_csv(report_path)
    print("\n=== MÉTRICAS ===")
    print(df)

    # --- Funciones de visualización ---
    def plot_bar_metric(df_local, metric_col, title, fname):
        modelos = df_local["modelo"].tolist()
        valores = df_local[metric_col].tolist()
        plt.figure()
        plt.bar(modelos, valores)
        plt.title(title)
        plt.ylabel(metric_col)
        plt.tight_layout()
        out_path = os.path.join(GRAF_DIR, fname)
        plt.savefig(out_path, dpi=150)
        plt.close()
        print("Guardado:", out_path)

    def plot_confusion(cm_local, labels, title, fname):
        plt.figure()
        plt.imshow(cm_local, cmap="Blues")
        plt.title(title)
        plt.xlabel("Predicción")
        plt.ylabel("Real")
        plt.xticks(range(len(labels)), labels)
        plt.yticks(range(len(labels)), labels)
        for i in range(cm_local.shape[0]):
            for j in range(cm_local.shape[1]):
                plt.text(j, i, str(cm_local[i, j]), ha="center", va="center")
        plt.tight_layout()
        out_path = os.path.join(GRAF_DIR, fname)
        plt.savefig(out_path, dpi=150)
        plt.close()
        print("Guardado:", out_path)

    # Barras
    plot_bar_metric(df, "f1_test",   "F1 (test) - Transformers",       "f1_test_transformers.png")
    plot_bar_metric(df, "prec_test", "Precisión (test) - Transformers","precision_test_transformers.png")
    plot_bar_metric(df, "rec_test",  "Recall (test) - Transformers",   "recall_test_transformers.png")
    plot_bar_metric(df, "acc_test",  "Accuracy (test) - Transformers", "accuracy_test_transformers.png")

    # Matriz de confusión
    cm_path = os.path.join(REPORT_DIR, cm_name)
    if os.path.exists(cm_path):
        cm_loaded = np.load(cm_path)
        plot_confusion(cm_loaded, ["Real", "Fake"], "Matriz de confusión - BETO", "cm_beto.png")

    # Curvas de aprendizaje de BETO
    plot_beto_curves("history_beto.npy")


if __name__ == "__main__":
    main()


Mounted at /content/drive
Usando GPU: True
GPU 0: Tesla T4 (UUID: GPU-55010e02-37e9-9e7f-2e40-ef690a84eeab)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

/tmp/ipython-input-78825684.py:382: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  pred_trainer = Trainer(model=best_model, tokenizer=tokenizer)


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: axeljasegundo (axeljasegundo-upiit) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



 Reporte guardado en: /content/drive/MyDrive/COLAB_ARCHIVOS/MET_Proyecto/Data/Salidas/reportes_transformers/report_transformers.csv
           modelo  acc_train  prec_train  rec_train  f1_train  acc_test  \
0  beto_finetuned   0.994286    0.995702   0.992857  0.994278    0.8475   

   prec_test  rec_test  f1_test   tn  fp  fn   tp      cm_path  best_lr  \
0   0.845771      0.85  0.84788  169  31  30  170  cm_beto.npy  0.00005   

   best_batch_size                                    model_dir  
0                8  modelos/Transformers/beto_lr5e-05_bs8/final  
 Mejor modelo en: /content/drive/MyDrive/COLAB_ARCHIVOS/MET_Proyecto/Data/Salidas/modelos/Transformers/beto_lr5e-05_bs8/final

=== MÉTRICAS ===
           modelo  acc_train  prec_train  rec_train  f1_train  acc_test  \
0  beto_finetuned   0.994286    0.995702   0.992857  0.994278    0.8475   

   prec_test  rec_test  f1_test   tn  fp  fn   tp      cm_path  best_lr  \
0   0.845771      0.85  0.84788  169  31  30  170  cm_beto.npy 